# Финальный воспроизводимый пакет SBMTeam

**Команда Kaggle:** SBMTeam  
**Участники:** Peretiatko Vsevolod (`sevaperetiatko`) и Platon Usachev (`omnistudent`)  
**Основной submission:** `submission_NEW5_sqrt_s100_aux20.csv`  
**Дополнительный финальный вариант:** `submission_rescue_corrected_best.csv`

В этом ноутбуке оставлен только воспроизводимый финальный конвейер: подготовка признаков, обучение двух финальных ансамблей, несколько содержательных базовых моделей для сравнения, расчёт validation ROC-AUC и сохранение файлов предсказаний. Поисковые эксперименты и Optuna-подбор сюда не перенесены, чтобы запуск укладывался в ограничения финального пакета.


## Условия воспроизводимости

- Все данные и служебные файлы читаются из папки проекта по относительным путям. Для запуска не требуется интернет или дополнительные локальные файлы вне пакета.
- Для воспроизводимости используется один фиксированный `seed` во всех компонентах, где присутствует случайность.
- Модели обучаются только на `train.csv`. Предоставленный `validation.csv` используется исключительно для итоговой оценки уже зафиксированных конфигураций и не добавляется к обучающей выборке.
- Преобразования признаков, параметры которых зависят от данных, обучаются только на `train`. Для обучающей выборки supervised-признаки строятся out-of-fold, поэтому объект не использует собственный `target` при построении своего признакового описания. Для `validation` и `test` применяются преобразования, обученные на `train`.
- Все гиперпараметры, наборы признаков и веса ансамблей зафиксированы заранее по результатам исследовательского этапа. В этом ноутбуке дополнительный подбор параметров не выполняется.
- После полного запуска создаются три файла: `final_submission.csv`, `submission_NEW5_sqrt_s100_aux20.csv` и `submission_rescue_corrected_best.csv`.


In [1]:
import os
from pathlib import Path

import pandas as pd
from IPython.display import display

from final_model_runner import MODEL_CONFIGURATIONS, SEED, run

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 240)

print("Зафиксированный seed:", SEED)
print("Рабочая папка:", Path.cwd())
print("Доступных CPU-ядер:", os.cpu_count())


Зафиксированный seed: 1027309
Рабочая папка: c:\Users\peret\Desktop\Third year\ML\comp\comp 1\sbmt_kaggle\SBMTeam_final_package
Доступных CPU-ядер: 12


## Дополнительные признаки

Помимо 512 исходных признаков, для части моделей строятся несколько дополнительных представлений объекта. Их цель - описать данные не только через отдельные координаты, но и через более общие характеристики: форму вектора, положение относительно двух классов, локальное окружение и основные направления изменчивости.

Дополнительные признаки разделены на несколько блоков.

### Агрегатные признаки

Эти признаки кратко описывают распределение значений внутри одного объекта.

| Признак | Что показывает |
|---|---|
| `total` | Сумму всех координат объекта |
| `max` | Наибольшую координату |
| `log1p_sum` | Сумму дополнительно логарифмированных координат; похожа на `total`, но меньше зависит от отдельных очень больших значений |
| `count_zeros` | Число нулевых координат |
| `relative_std` | Насколько неравномерно распределены значения между координатами относительно их среднего уровня |
| `max_share` | Долю максимальной координаты в общей сумме - показывает, сосредоточена ли значительная часть объекта в одной координате. |

### Геометрические признаки

Этот блок описывает положение объекта относительно обучающих примеров классов 0 и 1. Вместо анализа каждой координаты отдельно используются расстояния, направления и проекции в пространстве признаков.

| Признак или группа | Что показывает |
|---|---|
| `d_euclid_diff` | Разницу евклидовых расстояний до средних объектов двух классов. Показывает, к центру какого класса объект расположен ближе. |
| `proj_w` | Положение объекта вдоль направления, соединяющего центры двух классов. |
| `cos_w` | Насколько направление на объект согласуется с направлением между центрами классов. |
| `lda_score` | Положение объекта вдоль линейного направления, которое лучше всего разделяет два класса. |
| `d_mahal_*`, `mahal_*` | Расстояния до классов с учётом различного масштаба и корреляций признаков. В отличие от обычного евклидова расстояния, разные направления пространства имеют разную значимость. |
| `rda_diff_*`, `rda_norm_diff_*` | Несколько вариантов сравнения расстояний до классов с различными оценками ковариации. Нормированные версии дополнительно уменьшают влияние общего масштаба расстояний. |

### Признаки плотности

Здесь оценивается не просто расстояние до центра класса, а то, насколько объект в целом похож на распределение каждого класса.

| Признак или группа | Что показывает |
|---|---|
| `qda_score` | Оценку квадратичного дискриминанта. В отличие от линейного LDA, позволяет двум классам иметь разные формы и ковариации распределений. |
| `pca_subspace_diff` | Сравнивает ошибки восстановления объекта двумя PCA-моделями, отдельно обученными на каждом классе. Меньшая ошибка означает, что объект лучше описывается подпространством соответствующего класса. |

### kNN-признаки

Эти признаки описывают локальное окружение объекта среди обучающих примеров.

| Признак | Что показывает |
|---|---|
| `knn_dist_diff_k15` | Сравнивает среднее расстояние до ближайших соседей класса 0 и класса 1. Показывает, к объектам какого класса текущий объект расположен ближе. |
| `knn_prob_class1_k15` | Долю объектов класса 1 среди ближайших соседей. Значение около 1 означает преимущественно окружение класса 1, около 0 - класса 0. |

### Координатные признаки

Для каждой исходной координаты оценивается, какие её значения чаще встречаются у класса 0 и у класса 1. Затем информация по координатам объединяется в несколько суммарных признаков.

| Признак или группа | Что показывает |
|---|---|
| `coord_llr_total` | Суммарное свидетельство всех координат в пользу одного из двух классов. |
| `coord_llr_monotonic_total` | Та же идея, но после дополнительного монотонного сглаживания зависимости между значением координаты и классом. |
| групповые `coord_llr_*` | Аналогичные суммы, рассчитанные отдельно для нескольких групп координат. Позволяют сохранить информацию о том, в какой части исходного вектора появился сигнал. |

### PCA128

`PCA128` - это компактное линейное представление исходных 512 признаков.

Признаки `pc_0, ..., pc_127` являются координатами объекта в 128 главных направлениях данных. PCA сохраняет направления с наибольшей изменчивостью и одновременно уменьшает размерность с 512 до 128 признаков.

### Преобразования блоков

Большинство engineered-признаков строится на `log1p`-преобразованных исходных координатах. Перед PCA исходные признаки дополнительно стандартизируются.

В названиях некоторых представлений встречаются суффиксы:

- `geometry_white16` - блок geometry сжат до 16 компонент PCA и whitened;
- `density_white5` - блок density сжат до 5 компонент;
- `aggregates_white4` - агрегатный блок сжат до 4 компонент;
- `coord_white4` - координатный блок сжат до 4 компонент.

Whitening здесь означает дополнительное масштабирование PCA-компонент так, чтобы направления с разной исходной дисперсией имели сопоставимый масштаб. Эти представления используются в `Corrected K0` и связанных с ним моделях.

Отдельно модель `S100` использует другое представление исходных признаков: вместо `log1p(raw)` берётся `sqrt(raw)`, после чего признаки стандартизируются и используются для построения основного RBF-ядра.

Некоторые дополнительные признаки зависят от `target` обучающей выборки, например расстояния до классов, LDA/QDA, PPCA, kNN и coordinate likelihood. Для объектов `train` такие признаки строятся out-of-fold: признаки каждого объекта рассчитываются с помощью преобразования, которое не обучалось на его собственном `target`. Для `validation` и `test` используются преобразования, обученные только на `train`.

## Зафиксированные конфигурации моделей

Для сравнения используются десять заранее выбранных конфигураций.

Первые четыре модели служат базовыми ориентирами: одно дерево решений, случайный лес, градиентный бустинг и обычный RBF SVC только на исходных признаках. Следующие четыре показывают развитие основной SVC-архитектуры и роль отдельных представлений данных. Последние две строки соответствуют двум финальным ансамблям.

Для каждой модели в таблице указано, чем она отличается от остальных, какие признаки использует и какие основные гиперпараметры были зафиксированы. Во время этого финального запуска параметры повторно не подбираются.


In [2]:
configuration_table = pd.DataFrame(MODEL_CONFIGURATIONS).rename(columns={
    "model": "Модель",
    "role": "Назначение",
    "parameters": "Параметры",
    "feature_set": "Признаки",
})

display(
    configuration_table.style
    .set_properties(**{
        "white-space": "pre-wrap",
        "text-align": "left",
        "vertical-align": "top",
    })
    .set_table_styles([
        {"selector": "th", "props": [("text-align", "left"), ("white-space", "nowrap")]},
        {"selector": "table", "props": [("width", "100%")]},
    ])
    .hide(axis="index")
)


Модель,Назначение,Параметры,Признаки
Дерево решений,"Первая простая модель. Показывает, насколько задача решается одним неглубоким деревом на полном наборе исходных и построенных признаков, без ансамблирования.","max_depth=5, min_samples_leaf=20","512 исходных признаков + geometry, density, aggregates, coord, kNN и PCA128; для части engineered-блоков используются PCA-whitened представления"
Случайный лес,"Ансамбль независимых деревьев на том же расширенном наборе признаков. Проверяет, насколько усреднение большого числа деревьев улучшает результат по сравнению с одиночным деревом.","n_estimators=1000, max_depth=8, min_samples_leaf=40, min_samples_split=80, max_features=sqrt","512 исходных признаков + geometry, density, aggregates, coord, kNN и PCA128"
Градиентный бустинг (GBDT),"Последовательный ансамбль неглубоких деревьев. Использует тот же набор признаков, что и другие древесные модели, и служит альтернативой RBF-SVC с другим способом моделирования нелинейных зависимостей.","n_estimators=2000, learning_rate=0.01, max_depth=3, min_samples_leaf=80, subsample=0.7","512 исходных признаков + geometry, density, aggregates, coord, kNN и PCA128"
Простой raw RBF SVC,"SVC без дополнительных engineered-признаков. Нужна, чтобы отдельно оценить качество самого RBF-классификатора и затем сравнить его с multi-kernel моделями.","log1p → StandardScaler; RBF SVC, C=10, gamma=scale",только 512 исходных признаков
Fine6 SVC,"Multi-kernel SVC. Для разных групп признаков строятся отдельные RBF-ядра, после чего они складываются с фиксированными весами. Основной вклад даёт raw-блок, а engineered-блоки дополняют его.","C=6; веса ядер: raw=.68, geometry=.10, density=.10, aggregates=.05, kNN=.05, coord=.02","log1p(raw) + исходные geometry, density, aggregates, coord и kNN-блоки"
Corrected K0 SVC,"Улучшенная версия Fine6. Общая multi-kernel структура и веса блоков сохраняются, но geometry, density, aggregates и coord заменены на PCA-whitened представления, а параметры RBF-ядер зафиксированы.","C=3; веса: raw=.68, geometry=.10, density=.10, aggregates=.05, kNN=.05, coord=.02",log1p(raw) + geometry_white16 + density_white5 + aggregates_white4 + coord_white4 + kNN
S100 SVC,"Вариант K0 с другим представлением исходных 512 признаков. В основном raw-ядре вместо log1p используется sqrt-преобразование; остальные engineered-блоки остаются такими же, как в corrected K0.","C=3; веса: sqrt_raw=.68, geometry=.10, density=.10, aggregates=.05, kNN=.05, coord=.02",sqrt(raw) + geometry_white16 + density_white5 + aggregates_white4 + coord_white4 + kNN
Вспомогательный clean_aux SVC,"SVC с ослабленными признаками. Отдельно она слабее основной модели, но её ранжирование отличается от Fine6/K0/S100, поэтому модель используется как дополнительный компонент финальных ансамблей для регуляризации.","C=30; веса ядер: PCA128=.75, aggregates_white4=.25",PCA128 + aggregates_white4
Rescue T6,"Второй финальный ансамбль. Объединяет старую Fine6, улучшенную corrected K0 и clean_aux модель. Смешиваются не исходные SVC-score, а их ранги.","rank blend: Fine6=.40, corrected K0=.40, clean_aux=.20","ранги предсказаний Fine6 SVC, corrected K0 SVC и clean_aux SVC"
NEW5 (финальная),"Финальный ансамбль с sqrt-представлением raw-признаков. Основную часть прогноза даёт S100, а clean_aux добавляет независимый PCA/aggregate-сигнал.","rank blend: S100=.80, clean_aux=.20",ранги предсказаний S100 SVC и clean_aux SVC


### Как устроены два финальных ансамбля

Обе финальные модели объединяют несколько SVC на уровне предсказаний.

Сначала для каждой SVC вычисляется `decision_function`. Затем значения внутри каждого набора предсказаний заменяются их процентильными рангами. После этого ранги отдельных моделей складываются с фиксированными весами.

Такое объединение используется потому, что разные SVC могут выдавать score в разных числовых масштабах, тогда как для ROC-AUC важен прежде всего порядок объектов. Переход к рангам позволяет объединять модели по их ранжированию, не подгоняя масштабы `decision_function`.

Все веса были выбраны на предыдущем экспериментальном этапе и в этом ноутбуке не изменяются.

In [3]:
FINAL_ENSEMBLES = {
    "NEW5": {
        "S100 SVC": 0.80,
        "clean_aux SVC": 0.20,
    },
    "Rescue T6": {
        "Fine6 SVC": 0.40,
        "Corrected K0 SVC": 0.40,
        "clean_aux SVC": 0.20,
    },
}

display(
    pd.DataFrame(FINAL_ENSEMBLES).fillna(0.0)
    .rename_axis("Компонент")
    .style.format("{:.2f}")
)


,NEW5,Rescue T6
Компонент,,
S100 SVC,0.80,0.00
clean_aux SVC,0.20,0.20
Fine6 SVC,0.00,0.40
Corrected K0 SVC,0.00,0.40


## Обучение, оценка и генерация submission

Функция `run()` выполняет весь финальный эксперимент от загрузки данных до сохранения submission-файлов.

Сначала загружаются `train.csv`, `validation.csv`, `test.csv` и `sample_submission.csv`. Все преобразования признаков обучаются на `train`, после чего теми же преобразованиями независимо обрабатываются `validation` и `test`. Затем на `train` обучаются десять зафиксированных конфигураций из таблицы выше.

Для каждой модели рассчитывается ROC-AUC на `validation.csv`. Validation используется только для итогового сравнения уже выбранных моделей: его объекты и метки не участвуют в обучении моделей или преобразований признаков.

После оценки финальные ансамбли применяются к `test.csv`, а полученные score записываются в submission-файлы. Дополнительно сохраняется время, затраченное на основные этапы конвейера.

Древесные baseline-модели получают полный набор исходных и engineered-признаков. Простой raw RBF SVC использует только исходные 512 признаков, поэтому его результат показывает, какой прирост дают дополнительные представления и multi-kernel архитектура.


In [4]:
result = run(output_dir=Path("."))

print("Созданы файлы:")
for output_path in result["output_paths"]:
    print(" -", output_path)


Созданы файлы:
 - submission_NEW5_sqrt_s100_aux20.csv
 - submission_rescue_corrected_best.csv
 - final_submission.csv


## Итоговое сравнение десяти моделей

Ниже приведён итоговый результат одного воспроизводимого запуска. Для каждой конфигурации показаны ROC-AUC на `validation.csv`, время выполнения и использованный seed. Описание архитектуры, признаков и гиперпараметров приведено в таблице выше и здесь не дублируется.


In [5]:
experiment_table = result["experiment_table"].copy()

display(
    experiment_table.style
    .format({
        "Validation ROC-AUC": "{:.6f}",
        "Время, сек": "{:.2f}",
    })
    .hide(axis="index")
)

timing_names = {
    "preprocessing_seconds": "Загрузка данных и построение признаков",
    "final_components_seconds": "Обучение компонентов финальных ансамблей",
    "comparison_models_seconds": "Обучение и оценка моделей из сравнительной таблицы",
    "test_prediction_seconds": "Предсказание на test и подготовка submission",
    "total_seconds": "Общее время выполнения run()",
}
timing_table = pd.DataFrame([
    {"Этап": timing_names.get(name, name), "Время, сек": seconds}
    for name, seconds in result["timings"].items()
])
display(timing_table.style.format({"Время, сек": "{:.2f}"}).hide(axis="index"))


Модель,Validation ROC-AUC,"Время, сек",Seed
Дерево решений,0.904298,1.31,1027309
Случайный лес,0.923661,5.53,1027309
Градиентный бустинг (GBDT),0.948776,165.59,1027309
Простой raw RBF SVC,0.947423,2.49,1027309
Fine6 SVC,0.955038,27.81,1027309
Corrected K0 SVC,0.954082,27.81,1027309
S100 SVC,0.955038,14.98,1027309
Вспомогательный clean_aux SVC,0.938342,27.81,1027309
Rescue T6,0.955179,347.12,1027309
NEW5 (финальная),0.954936,347.12,1027309


Этап,"Время, сек"
Загрузка данных и построение признаков,128.96
rescue_components_fit_predict_seconds,27.81
s100_fit_predict_seconds,14.98
Общее время выполнения run(),347.12


## Версии среды

Таблица фиксирует версии Python и основных библиотек, от которых зависит воспроизводимость расчётов.


In [6]:
versions_table = pd.DataFrame(
    [{"Пакет": name, "Версия": version} for name, version in result["versions"].items()]
)
display(versions_table.style.hide(axis="index"))


Пакет,Версия
python,3.14.2
numpy,2.5.3
pandas,3.0.6
scipy,1.18.1
scikit-learn,1.9.1
joblib,1.6.0


## Автоматические проверки результата

В конце проверяется, что `row_id` совпадает с test в исходном порядке, вероятности конечны и находятся в диапазоне `[0, 1]`, а все три ожидаемых CSV действительно созданы. Эти проверки ловят наиболее опасные ошибки сборки submission до загрузки на Kaggle или Moodle.


In [7]:
assert result["row_id_exact"], "row_id в submission не совпадает с test.csv"
assert result["probabilities_valid"], "Предсказания содержат некорректные вероятности"
assert all(Path(path).is_file() for path in result["output_paths"]), "Созданы не все CSV-файлы"

print("Проверка row_id: пройдена")
print("Проверка вероятностей: пройдена")
print("Все ожидаемые CSV созданы: да")


Проверка row_id: пройдена
Проверка вероятностей: пройдена
Все ожидаемые CSV созданы: да
